# DeepSeek-V3 from scratch

A small, readable PyTorch implementation of the **DeepSeek-V3** architecture in a
single notebook. Run the cells top to bottom.

The three things that make it DeepSeek-V3:

1. **MLA (Multi-head Latent Attention)** — keys/values are compressed to a small
   latent and up-projected per head; each head splits into a content part (no
   position) and a small decoupled part that carries RoPE.
2. **DeepSeekMoE** — always-on *shared* experts + a large pool of *routed*
   experts, selected by a **sigmoid** gate with **auxiliary-loss-free** balancing.
3. **MTP (Multi-Token Prediction)** — extra heads that predict additional future
   tokens during training (dropped at inference).

The config holds the **real** DeepSeek-V3 (671B) values. The final cell runs a
forward pass on a **small** config so the notebook executes on CPU.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, replace

## 1. Config

Real DeepSeek-V3 defaults (from the official `config.json`). Each per-head query/
key is `qk_nope_head_dim + qk_rope_head_dim` = 128 + 64 = 192 dims wide.

In [ ]:
@dataclass
class DeepSeekV3Config:
    # Vocab / embedding
    vocab_size: int = 129280

    # Transformer backbone
    hidden_size: int = 7168
    num_hidden_layers: int = 61
    num_attention_heads: int = 128

    # Multi-head Latent Attention (MLA)
    q_lora_rank: int = 1536             # query down-projection (latent) dim
    kv_lora_rank: int = 512             # key/value latent dim
    qk_nope_head_dim: int = 128         # per-head content (no-position) dim
    qk_rope_head_dim: int = 64          # per-head decoupled RoPE dim
    v_head_dim: int = 128               # per-head value dim

    # Mixture of Experts (DeepSeekMoE)
    intermediate_size: int = 18432      # dense FFN hidden size
    moe_intermediate_size: int = 2048   # per-expert hidden size
    n_routed_experts: int = 256
    n_shared_experts: int = 1
    num_experts_per_tok: int = 8        # top-k routed experts
    first_k_dense_replace: int = 3      # first N layers are dense FFN
    n_group: int = 8                    # expert groups for routing
    topk_group: int = 4                 # groups kept per token
    routed_scaling_factor: float = 2.5
    norm_topk_prob: bool = True

    # Multi-Token Prediction (MTP)
    num_mtp_modules: int = 1

    # Normalisation / RoPE
    rms_norm_eps: float = 1e-6
    rope_theta: float = 10000.0
    max_position_embeddings: int = 163840   # 4096 base, extended via YaRN

    @property
    def qk_head_dim(self) -> int:
        """Total per-head query/key dim = content (nope) + decoupled RoPE."""
        return self.qk_nope_head_dim + self.qk_rope_head_dim

## 2. RMSNorm

Normalise by the root-mean-square of the activations (no mean subtraction, no
bias). Computed in fp32 for stability.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, hidden_size: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.eps = eps

    def forward(self, x):
        dtype = x.dtype
        x = x.float()
        variance = x.pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(variance + self.eps)
        return self.weight * x.to(dtype)

## 3. Rotary Position Embedding (RoPE)

RoPE is applied only to the small *decoupled* slice of the query/key. The
`_deinterleave` step matches DeepSeek-V3 exactly so the half-split `rotate_half`
lands on the right frequency pairs.

In [ ]:
class RotaryEmbedding(nn.Module):
    """Precomputes cos/sin tables for the decoupled RoPE dimension."""

    def __init__(self, dim: int, base: float = 10000.0, max_position: int = 2048):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.max_position = max_position

    def forward(self, seq_len, device, dtype):
        t = torch.arange(seq_len, device=device, dtype=torch.float32)
        freqs = torch.outer(t, self.inv_freq.to(device))   # (seq_len, dim/2)
        emb = torch.cat((freqs, freqs), dim=-1)             # (seq_len, dim)
        return emb.cos().to(dtype), emb.sin().to(dtype)


def rotate_half(x):
    """Rotates the two halves of the last dimension: [-x2, x1]."""
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def _deinterleave(x):
    """[x0,x1,x2,x3,...] -> [x0,x2,...,x1,x3,...] (matches DeepSeek-V3)."""
    b, h, s, d = x.shape
    return x.view(b, h, s, d // 2, 2).transpose(4, 3).reshape(b, h, s, d)


def apply_rotary(x, cos, sin):
    """Apply RoPE to x of shape (batch, heads, seq, rope_dim)."""
    x = _deinterleave(x)
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    return x * cos + rotate_half(x) * sin

## 4. SwiGLU MLP

Used both as the dense FFN in the first layers and as each individual expert:
`down_proj( silu(gate_proj(x)) * up_proj(x) )`.

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_size: int, intermediate_size: int):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

## 5. Multi-head Latent Attention (MLA)

Instead of storing full keys/values, MLA compresses them into a small latent
(`kv_lora_rank`) that is up-projected per head on the fly; the query comes from
its own latent (`q_lora_rank`). Each head's query/key = a content (`nope`) part
with no position info + a decoupled `rope` part that carries RoPE. The decoupled
key is shared across all heads. This implements full-sequence causal attention
(no KV cache) for simplicity.

In [ ]:
class MLA(nn.Module):
    def __init__(self, config: DeepSeekV3Config):
        super().__init__()
        self.num_heads = config.num_attention_heads
        self.qk_nope_head_dim = config.qk_nope_head_dim
        self.qk_rope_head_dim = config.qk_rope_head_dim
        self.qk_head_dim = config.qk_head_dim          # nope + rope
        self.v_head_dim = config.v_head_dim
        self.kv_lora_rank = config.kv_lora_rank

        # Query path: hidden -> latent -> per-head q (nope + rope)
        self.q_down_proj = nn.Linear(config.hidden_size, config.q_lora_rank, bias=False)
        self.q_norm = RMSNorm(config.q_lora_rank, eps=config.rms_norm_eps)
        self.q_up_proj = nn.Linear(config.q_lora_rank, self.num_heads * self.qk_head_dim, bias=False)

        # Key/Value path: hidden -> [latent | shared rope key]
        self.kv_down_proj = nn.Linear(config.hidden_size, self.kv_lora_rank + self.qk_rope_head_dim, bias=False)
        self.kv_norm = RMSNorm(self.kv_lora_rank, eps=config.rms_norm_eps)
        self.kv_up_proj = nn.Linear(self.kv_lora_rank, self.num_heads * (self.qk_nope_head_dim + self.v_head_dim), bias=False)

        # Output projection
        self.o_proj = nn.Linear(self.num_heads * self.v_head_dim, config.hidden_size, bias=False)

        self.rotary = RotaryEmbedding(self.qk_rope_head_dim, base=config.rope_theta,
                                      max_position=config.max_position_embeddings)
        self.softmax_scale = self.qk_head_dim ** -0.5

    def forward(self, hidden_states):
        B, T, _ = hidden_states.shape

        # Query: down -> norm -> up -> split into nope/rope
        q = self.q_up_proj(self.q_norm(self.q_down_proj(hidden_states)))
        q = q.view(B, T, self.num_heads, self.qk_head_dim).transpose(1, 2)
        q_nope, q_rope = torch.split(q, [self.qk_nope_head_dim, self.qk_rope_head_dim], dim=-1)

        # Key/Value: down -> split latent & shared rope key
        kv = self.kv_down_proj(hidden_states)
        c_kv, k_rope = torch.split(kv, [self.kv_lora_rank, self.qk_rope_head_dim], dim=-1)
        k_rope = k_rope.view(B, T, 1, self.qk_rope_head_dim).transpose(1, 2)   # shared across heads

        kv = self.kv_up_proj(self.kv_norm(c_kv))
        kv = kv.view(B, T, self.num_heads, self.qk_nope_head_dim + self.v_head_dim).transpose(1, 2)
        k_nope, value = torch.split(kv, [self.qk_nope_head_dim, self.v_head_dim], dim=-1)

        # RoPE on the decoupled parts only
        cos, sin = self.rotary(T, hidden_states.device, hidden_states.dtype)
        q_rope = apply_rotary(q_rope, cos, sin)
        k_rope = apply_rotary(k_rope, cos, sin)
        k_rope = k_rope.expand(B, self.num_heads, T, self.qk_rope_head_dim)

        # Assemble full query/key and run causal attention
        query = torch.cat([q_nope, q_rope], dim=-1)
        key = torch.cat([k_nope, k_rope], dim=-1)

        scores = torch.matmul(query, key.transpose(-1, -2)) * self.softmax_scale
        causal = torch.triu(torch.full((T, T), float("-inf"), device=scores.device), diagonal=1)
        scores = scores + causal
        attn = F.softmax(scores.float(), dim=-1).to(query.dtype)

        out = torch.matmul(attn, value)
        out = out.transpose(1, 2).reshape(B, T, self.num_heads * self.v_head_dim)
        return self.o_proj(out)

## 6. DeepSeekMoE

**Shared + routed experts.** Shared experts always run; only
`num_experts_per_tok` routed experts are selected per token.

**Auxiliary-loss-free load balancing.** A `sigmoid` gate produces affinities. A
per-expert bias is added *only* for the top-k selection decision — the weights
used to combine experts come from the **unbiased** scores. Routing is also
*group-limited*: pick the `topk_group` strongest groups first, then experts
within them.

In [ ]:
class MoEGate(nn.Module):
    """Which routed experts each token goes to, and with what weight."""

    def __init__(self, config: DeepSeekV3Config):
        super().__init__()
        self.top_k = config.num_experts_per_tok
        self.n_routed_experts = config.n_routed_experts
        self.n_group = config.n_group
        self.topk_group = config.topk_group
        self.norm_topk_prob = config.norm_topk_prob
        self.routed_scaling_factor = config.routed_scaling_factor

        self.weight = nn.Parameter(torch.empty(self.n_routed_experts, config.hidden_size))
        nn.init.normal_(self.weight, std=0.02)
        # Aux-loss-free balancing bias (selection-only; not trained by grad here).
        self.register_buffer("e_score_correction_bias", torch.zeros(self.n_routed_experts))

    def forward(self, x):
        logits = F.linear(x.float(), self.weight.float())   # (n, n_experts)
        scores = logits.sigmoid()                           # V3 uses sigmoid

        # bias added for SELECTION only
        scores_for_choice = scores + self.e_score_correction_bias

        # group-limited routing: group strength = sum of its top-2 experts
        n = x.shape[0]
        experts_per_group = self.n_routed_experts // self.n_group
        grouped = scores_for_choice.view(n, self.n_group, experts_per_group)
        group_scores = grouped.topk(min(2, experts_per_group), dim=-1)[0].sum(-1)
        group_idx = group_scores.topk(self.topk_group, dim=-1)[1]
        group_mask = torch.zeros_like(group_scores).scatter_(1, group_idx, 1.0)
        score_mask = group_mask.unsqueeze(-1).expand(n, self.n_group, experts_per_group).reshape(n, self.n_routed_experts)
        masked = scores_for_choice.masked_fill(score_mask == 0, float("-inf"))

        # top-k expert selection; weights come from UNBIASED sigmoid scores
        topk_idx = masked.topk(self.top_k, dim=-1)[1]
        topk_weight = scores.gather(1, topk_idx)

        if self.top_k > 1 and self.norm_topk_prob:
            topk_weight = topk_weight / (topk_weight.sum(-1, keepdim=True) + 1e-20)
        topk_weight = topk_weight * self.routed_scaling_factor
        return topk_idx, topk_weight.to(x.dtype)


class DeepSeekMoE(nn.Module):
    def __init__(self, config: DeepSeekV3Config):
        super().__init__()
        self.top_k = config.num_experts_per_tok
        self.gate = MoEGate(config)
        self.experts = nn.ModuleList(
            MLP(config.hidden_size, config.moe_intermediate_size)
            for _ in range(config.n_routed_experts)
        )
        # Shared expert(s): always applied, intermediate scaled by count.
        self.shared_experts = MLP(config.hidden_size, config.moe_intermediate_size * config.n_shared_experts)

    def forward(self, hidden_states):
        B, T, H = hidden_states.shape
        x = hidden_states.view(-1, H)                # (n_tokens, hidden)
        topk_idx, topk_weight = self.gate(x)

        out = torch.zeros_like(x)
        flat_idx = topk_idx.view(-1)
        flat_w = topk_weight.view(-1, 1)
        token_ids = torch.arange(x.shape[0], device=x.device).repeat_interleave(self.top_k)
        for e, expert in enumerate(self.experts):
            mask = flat_idx == e
            if not mask.any():
                continue
            sel = token_ids[mask]
            out.index_add_(0, sel, expert(x[sel]) * flat_w[mask])

        out = out + self.shared_experts(x)           # shared expert: ungated
        return out.view(B, T, H)

## 7. Decoder layer

Pre-norm transformer block. The first `first_k_dense_replace` layers use a dense
MLP; the rest use MoE.

    h = h + MLA(input_layernorm(h))
    h = h + FFN(post_attention_layernorm(h))

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, config: DeepSeekV3Config, layer_idx: int):
        super().__init__()
        self.self_attn = MLA(config)
        self.input_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

        self.is_moe = layer_idx >= config.first_k_dense_replace
        if self.is_moe:
            self.mlp = DeepSeekMoE(config)
        else:
            self.mlp = MLP(config.hidden_size, config.intermediate_size)

    def forward(self, hidden_states):
        hidden_states = hidden_states + self.self_attn(self.input_layernorm(hidden_states))
        hidden_states = hidden_states + self.mlp(self.post_attention_layernorm(hidden_states))
        return hidden_states

## 8. Multi-Token Prediction (MTP)

A training-time module that predicts an additional future token. It reuses the
main model's embedding and output head (shared), runs one transformer block:

    proj_in = M_k · [ RMSNorm(h_prev) ; RMSNorm(emb(next_token)) ]
    h_k     = TransformerBlock(proj_in)

In [ ]:
class MTPModule(nn.Module):
    def __init__(self, config: DeepSeekV3Config, layer_idx: int):
        super().__init__()
        self.hidden_norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.embed_norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.proj = nn.Linear(2 * config.hidden_size, config.hidden_size, bias=False)
        self.block = DecoderLayer(config, layer_idx)

    def forward(self, prev_hidden, token_embeds):
        x = torch.cat([self.hidden_norm(prev_hidden), self.embed_norm(token_embeds)], dim=-1)
        x = self.proj(x)
        return self.block(x)

## 9. Full model

`tokens -> embed -> [DecoderLayer x N] -> RMSNorm -> lm_head -> logits`. The MTP
modules share the embedding and output head with the main model.

In [ ]:
class DeepSeekV3Model(nn.Module):
    def __init__(self, config: DeepSeekV3Config):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList(DecoderLayer(config, i) for i in range(config.num_hidden_layers))
        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.mtp_modules = nn.ModuleList(
            MTPModule(config, layer_idx=config.num_hidden_layers + k)
            for k in range(config.num_mtp_modules)
        )

    def backbone(self, input_ids):
        hidden_states = self.embed_tokens(input_ids)
        for layer in self.layers:
            hidden_states = layer(hidden_states)
        return hidden_states

    def forward(self, input_ids, return_mtp: bool = False):
        hidden_states = self.backbone(input_ids)
        logits = self.lm_head(self.norm(hidden_states))
        if not return_mtp:
            return logits

        mtp_logits = []
        prev_hidden = hidden_states
        for k, mtp in enumerate(self.mtp_modules, start=1):
            shifted = torch.roll(input_ids, shifts=-k, dims=1)
            shifted[:, -k:] = 0
            token_embeds = self.embed_tokens(shifted)
            prev_hidden = mtp(prev_hidden, token_embeds)
            mtp_logits.append(self.lm_head(self.norm(prev_hidden)))
        return logits, mtp_logits

    def num_parameters(self):
        return sum(p.numel() for p in self.parameters())

## 10. Sanity check (small config)

The config defaults above are the **real** DeepSeek-V3 (671B) values, which need
many GPUs. To actually run the code on CPU, we shrink the sizes with
`dataclasses.replace` — the architecture is identical, only smaller.

In [ ]:
torch.manual_seed(0)

small = replace(
    DeepSeekV3Config(),
    vocab_size=1024, hidden_size=256, num_hidden_layers=4,
    num_attention_heads=8, q_lora_rank=96, kv_lora_rank=64,
    qk_nope_head_dim=32, qk_rope_head_dim=16, v_head_dim=32,
    intermediate_size=512, moe_intermediate_size=128,
    n_routed_experts=8, num_experts_per_tok=2, first_k_dense_replace=1,
    n_group=2, topk_group=1,
)

model = DeepSeekV3Model(small).eval()
print("parameters      :", f"{model.num_parameters():,}")
print("layer FFN types :", ["MoE" if l.is_moe else "dense" for l in model.layers])

input_ids = torch.randint(0, small.vocab_size, (2, 16))
with torch.no_grad():
    logits, mtp_logits = model(input_ids, return_mtp=True)

print("main logits     :", tuple(logits.shape))
print("MTP[1] logits   :", tuple(mtp_logits[0].shape))
assert logits.shape == (2, 16, small.vocab_size)
assert torch.isfinite(logits).all()
print("OK - forward pass produced finite logits.")